<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2009%20-%202026-05-05%20-%20SQL%20con%20Python/clase_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 9 · SQL con Python

Hoy juntan lo de toda la semana en un pipeline. Ingesta, limpieza, query, entrega. Y sí, aprenden SQL — no para convertirse en DBAs, sino para entender datos relacionales.

> **Hoy haces** · Montar el pipeline completo de datos: ingesta, limpieza, estructuración, tests y consultas SQL con SQLAlchemy. Es la S2. ~2h30.
>
> **Entrega** · **ENTREGA S2 — Pipeline de Datos.** Módulo de ingesta + limpieza + tests mínimos + DB local, antes de medianoche del martes 5 de mayo.

In [ ]:
import random, numpy as np, pandas as pd, sqlite3
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("Setup completo")

## Cargar Titanic en SQLite

¿Puedo responder preguntas con SQL en lugar de pandas? Veamos:

In [ ]:
import seaborn as sns

# Cargar dataset
titanic = sns.load_dataset("titanic").dropna(subset=["age", "fare"])

# Crear conexión SQLite en memoria
conn = sqlite3.connect(":memory:")

# Cargar DataFrame como tabla SQL
titanic.to_sql("titanic", conn, index=False, if_exists="replace")

# Test: contar filas
query = "SELECT COUNT(*) as total FROM titanic;"
resultado = pd.read_sql_query(query, conn)
print(f"Total pasajeros en DB: {resultado.iloc[0, 0]}")

`pd.read_sql_query()` ejecuta SQL y devuelve un DataFrame. La base está en memoria: súper rápido, sin I/O.

## SELECT y WHERE

¿Cuántos hombres sobrevivieron (sex="male" AND survived=1)? Completa la consulta:

In [ ]:
# Completa la consulta SQL
query = """
SELECT COUNT(*) as hombres_vivos
FROM titanic
WHERE sex = 'male' AND survived = ___;
"""

resultado = pd.read_sql_query(query, conn)
print(f"Hombres sobrevivientes: {resultado.iloc[0, 0]}")

In [ ]:
assert resultado.iloc[0, 0] > 0, "Deberías encontrar al menos 1"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# WHERE sex = 'male' AND survived = 1

¿Cómo escribirías esto con pandas? `df[(df["sex"] == "male") & (df["survived"] == 1)]`. ¿Cuál es más legible? Eso depende de ti. SQL es más explícito, pandas es más Pythónico.

## GROUP BY + agregaciones

Edad promedio, tarifa promedio y count por pclass. Escribe el GROUP BY:

In [ ]:
# Escribe el GROUP BY
query = """
SELECT pclass, COUNT(*) as pasajeros, AVG(age) as edad_promedio, AVG(fare) as tarifa_promedio
FROM titanic
___ pclass
___;
"""

resultado = pd.read_sql_query(query, conn)
print(resultado.round(2))

In [ ]:
assert resultado.shape[0] == 3, f"Esperaba 3 clases, obtuve {resultado.shape[0]}"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# GROUP BY pclass ORDER BY pclass;

¿Cómo harías esto con `groupby()` en pandas? `titanic.groupby("pclass")[["age", "fare"]].agg({"age": "mean", "fare": "mean"})`. Similar, verdad.

## JOIN + pipeline end-to-end

Sin skeleton. Tienes dos tablas (titanic + una tabla "destinos" que relaciona pclass con puerto). Pasos:

1. Crea tabla "puertos": pclass → puerto (simula datos).
2. Usa INNER JOIN para listar pasajeros con su puerto.
3. Cuenta pasajeros por puerto.
4. Calcula tasa supervivencia por puerto.

In [ ]:
# Tu código aquí
# Paso 1: crear tabla puertos
puertos_df = pd.DataFrame({
    "pclass": [1, 2, 3],
    "puerto": ["Southampton", "Southampton", "Cherbourg"]
})

puertos_df.to_sql("puertos", conn, index=False, if_exists="replace")

# Paso 2: JOIN
query_join = """
SELECT t.passengerid, t.pclass, p.puerto, t.survived, t.age
FROM titanic t
___ INNER JOIN puertos p ON t.pclass = p.pclass
___ 5;
"""

resultado_join = pd.read_sql_query(query_join, conn)
print(f"Filas después de JOIN: {len(resultado_join)}")
print(resultado_join.head())

# Paso 3: count por puerto
query_count = "SELECT puerto, COUNT(*) as total FROM titanic t INNER JOIN puertos p ON t.pclass = p.pclass GROUP BY puerto ORDER BY total DESC;"
resultado_count = pd.read_sql_query(query_count, conn)
print(f"\nPasajeros por puerto:")
print(resultado_count)

In [ ]:
# Criterios de aceptación
assert len(resultado_join) > 0, "Falta resultado_join"
assert "puerto" in resultado_join.columns, "Falta columna puerto"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# LIMIT
# ORDER BY

## SQL vs. Pandas

¿Cuál es más rápido? Para tablas pequeñas, son iguales. Para > 100MB, SQL gana.

In [ ]:
# SQL
sql_result = pd.read_sql_query("SELECT sex, AVG(age) as edad_promedio FROM titanic GROUP BY sex;", conn)

# Pandas
pandas_result = titanic.groupby("sex")["age"].mean().reset_index()
pandas_result.columns = ["sex", "edad_promedio"]

print("SQL:")
print(sql_result)
print("\nPandas:")
print(pandas_result)
print("\nResultados idénticos: ", sql_result.equals(pandas_result.round(10)))

Para datos pequeños, ambos son rápidos. El poder de SQL es cuando tienes millones de filas en una base de datos real. Allí, SQL filtra en el servidor antes de enviar datos a Python. Es más eficiente.

## Checkpoint

1. **SELECT…FROM…WHERE:** filtra filas según condición.
2. **GROUP BY + HAVING:** agrupa y filtra grupos.
3. **JOIN:** combina dos tablas por llave común.
4. **COUNT, AVG, SUM, MIN, MAX:** funciones de agregación.

Si algo no quedó claro, vuelve atrás.

In [ ]:
assert conn is not None, "Conexión no existe"
print("Checkpoint ✓ — puedes continuar")

## El pipeline de datos

```
CSV/JSON
  ↓
DataFrame (pandas)
  ↓
SQLite (en memoria o en disco)
  ↓
Consultas SQL (SELECT, WHERE, GROUP BY, JOIN)
  ↓
Resultados → de vuelta a DataFrame
  ↓
Visualización / Análisis final
```

## Para recordar

- **SQLite en memoria:** `conn = sqlite3.connect(":memory:")`
- **Cargar DataFrame:** `df.to_sql("tabla", conn, ...)`
- **Ejecutar SQL:** `pd.read_sql_query(query, conn)` → DataFrame
- **SELECT…WHERE…GROUP BY:** lo básico de SQL.
- **INNER JOIN:** combina dos tablas.
- **SQL vs. Pandas:** elige según contexto (tablas grandes → SQL, exploratorio → Pandas).

---

## ENTREGA SEMANA 2 — Pipeline Completo

**Tu equipo debe entregar un notebook `pipeline_semana2.ipynb` que:**

1. **Ingesta:** carga desde CSV/Excel/API o genera datos sintéticos
2. **Transformación (Pandas + SQL):**
   - Manejo de nulos, duplicados, tipos
   - Agregaciones y groupby
   - Al menos 1 consulta SQL
3. **Estadísticas:**
   - `.describe()`, percentiles
   - Detección de outliers (IQR)
4. **Visualización:**
   - Al menos 4 plots (histogramas, boxplots, scatter, bar)
5. **Validación (pytest):**
   - Al menos 3 asserts que verifiquen calidad

**Criterios de aceptación:**
- Corre de inicio a fin sin errores
- Todos los pasos documentados en markdown
- Variables derivadas con sentido de negocio
- Gráficos con título, labels y `tight_layout()`
- ≤ 800 líneas

## Reflexión

En tus propias palabras, explícale a un compañero **cuándo usarías SQL en lugar de pandas** en máximo 3 oraciones.

> **Recordatorio** · **ENTREGA S2 hoy.** Antes de medianoche del martes 5 de mayo en el repo del equipo.

---

## Para tu equipo — ENTREGA FINAL S2

**Qué entregar:**

- `pipeline_semana2.ipynb` (notebook con todo el análisis)
- `src/pipeline.py` (versión ejecutable con `python -m src.pipeline`)
- `requirements.txt` (librerías necesarias)
- `README.md` (explicación de qué hace el pipeline)
- Todos los archivos en un **repositorio Git**

**Deadline:** Hoy, martes 5 de mayo, antes de las 18:00 UTC-5

**Cómo entregar:**
- Push a su repo (preferentemente GitHub)
- Envían link en el formulario de Classroom
- Opcionalmente: presenten oralmente (5 min) su pipeline